In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_file = Path.cwd() / "secrets.env"
load_dotenv(env_file)

groq_key = os.getenv("GROQ_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not groq_key:
    raise RuntimeError("GROQ_API_KEY is missing. Add it to secrets.env, then restart and run this cell.")

In [3]:
from openai import OpenAI

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_groq import ChatGroq

C:\Users\usama\AppData\Local\Temp\ipykernel_11532\3847616107.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [6]:
client = OpenAI(
    api_key= groq_key,
    base_url= 'https://api.groq.com/openai/v1/'
)

In [7]:
def check_emotion(prompt):
    response = client.chat.completions.create(
        model = 'openai/gpt-oss-20b',
        messages= [
            {
                'role' : 'system',
                'content': f"you'll be given a sentence , check it's emotion"
            },
            {
                'role' : 'user',
                'content': "I'm so lonely"
            },
            {
                'role' : 'assistant',
                'content' : 'sad'
            },
            {
                'role' : 'user',
                'content': prompt
            }
        ]
    )
    return response.choices[0].message.content.strip()


In [8]:
print(check_emotion('funny world we live in '))

amused


In [58]:
text_loader = TextLoader(
    file_path= 'Games record .txt',
    encoding= 'utf-8'
)
raw_document = text_loader.load()

In [59]:
text_splitter = RecursiveCharacterTextSplitter()
splitted_docs = text_splitter.split_documents(raw_document)

In [60]:
embeddings = HuggingFaceEmbeddings(
     model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [61]:
vector_store = FAISS.from_documents(documents= splitted_docs, embedding= embeddings)

In [62]:
memory = ConversationBufferMemory(memory_key= "chat_history" , return_messages= True)

In [63]:
qa = ConversationalRetrievalChain.from_llm(
    ChatGroq(
        model= 'openai/gpt-oss-20b',
        api_key= groq_key,
        temperature= 0.7
    ),
    vector_store.as_retriever(),
    memory = memory
)

In [64]:
 query = 'When was GTA 4 added?'

In [65]:
print("qa.memory:", qa.memory)
print("memory key:", qa.memory.memory_key if qa.memory else None)
print(qa.prep_inputs({"question": "test"}))

qa.memory: chat_memory=InMemoryChatMessageHistory(messages=[]) return_messages=True memory_key='chat_history'
memory key: chat_history
{'question': 'test', 'chat_history': []}


In [66]:
result = qa.invoke({
    "question": query,
})

In [68]:
result['answer'].strip()

'GTA\u202f4 was added on **18‑4‑19** (April\u202f18,\u202f2019).'